# 15 Prepare Compustat and CCM

目标：执行 Compustat 年报筛选、锁定的 PS/SH/BE 回退规则及 CCM 唯一映射。

- 输入：compfunda.csv、ccm_linktable.csv
- 正式输出：compustat_annual_clean.parquet、ccm_unique_links.parquet、映射冲突审计
- 主键：gvkey+datadate；gvkey+linkdt+linkenddt
- 上游依赖：14
- smoke 模式仅使用真实本地数据子样本，并写入 `data/audit/smoke/`；不冒充正式结果。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import numpy as np
import pandas as pd
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.runtime_paths import output_path

n=None if not SMOKE else 300_000
comp=pd.read_csv(PATHS["compustat"],nrows=n,low_memory=False)
raw_n=len(comp)
comp=comp[(comp["consol"]=="C")&(comp["indfmt"]=="INDL")&(comp["datafmt"]=="STD")&(comp["curcd"]=="USD")].copy()
comp["gvkey"]=comp["gvkey"].astype(str).str.replace(r"\.0$","",regex=True).str.zfill(6)
comp["datadate"]=pd.to_datetime(comp["datadate"],errors="coerce")
numeric=[c for c in comp.columns if c not in {"gvkey","datadate","costat","curcd","datafmt","indfmt","consol","cusip"}]
comp[numeric]=comp[numeric].apply(pd.to_numeric,errors="coerce")
# PS: PSTKRV -> PSTKL -> PSTK; only all-missing PS is explicitly zero.
comp["PS"]=comp["pstkrv"].combine_first(comp["pstkl"]).combine_first(comp["pstk"]).fillna(0.0)
# SH: SEQ -> CEQ+PSTK -> AT-LT-MIB. Missing components are not silently zero.
comp["SH"]=comp["seq"].combine_first(comp["ceq"]+comp["PS"]).combine_first(comp["at"]-comp["lt"])
# BE: SH + TXDITC - PS; TXDITC alone may be zero when absent.
comp["BE"]=comp["SH"]+comp["txditc"].fillna(0.0)-comp["PS"]
comp.loc[comp["BE"]<=0,"BE"]=np.nan
comp=comp.sort_values(["gvkey","datadate"]).drop_duplicates(["gvkey","datadate"],keep="last")

ccm=pd.read_csv(PATHS["ccm"],nrows=n,low_memory=False,dtype=str)
ccm=ccm.rename(columns={"LPERMNO":"permno","LINKDT":"linkdt","LINKENDDT":"linkenddt","LINKPRIM":"linkprim","LINKTYPE":"linktype"})
ccm["gvkey"]=ccm["gvkey"].astype(str).str.replace(r"\.0$","",regex=True).str.zfill(6)
ccm["permno"]=pd.to_numeric(ccm["permno"],errors="coerce").astype("Int64")
linkdt_text=ccm["linkdt"].astype("string").str.strip()
linkend_text=ccm["linkenddt"].astype("string").str.strip()
ccm["linkdt"]=pd.to_datetime(linkdt_text.where(~linkdt_text.isin(["B",".B","",None])),errors="coerce").fillna(pd.Timestamp("1900-01-01"))
ccm["linkenddt"]=pd.to_datetime(linkend_text.where(~linkend_text.isin(["E",".E","",None])),errors="coerce").fillna(pd.Timestamp("2099-12-31"))
ccm=ccm[ccm["linkprim"].isin(["P","C"])&ccm["linktype"].isin(["LC","LU"])&ccm["permno"].notna()].copy()
ccm["prim_priority"]=ccm["linkprim"].map({"P":0,"C":1})
clean=ccm.sort_values(["gvkey","linkdt","linkenddt","linkprim","linktype","permno"]).drop_duplicates(
    ["gvkey","linkdt","linkenddt","linkprim","linktype","permno"])
# Apply the locked resolution at each actual Compustat statement date.
mapped=comp.merge(clean[["gvkey","permno","linkdt","linkenddt","linkprim","linktype","prim_priority"]],on="gvkey",how="left")
mapped=mapped[(mapped["permno"].notna())&(mapped["datadate"]>=mapped["linkdt"])&(mapped["datadate"]<=mapped["linkenddt"])].copy()
statement_key=["gvkey","datadate"]
mapped=mapped[mapped["prim_priority"].eq(mapped.groupby(statement_key)["prim_priority"].transform("min"))]
mapped=mapped[mapped["linkdt"].eq(mapped.groupby(statement_key)["linkdt"].transform("max"))]
conflict_keys=mapped.groupby(statement_key)["permno"].nunique().rename("n_permno").reset_index()
conflicts=mapped.merge(conflict_keys[conflict_keys["n_permno"]>1],on=statement_key,how="inner")
mapped_clean=mapped.merge(conflict_keys[conflict_keys["n_permno"]>1][statement_key],on=statement_key,how="left",indicator=True)
mapped_clean=mapped_clean[mapped_clean["_merge"]=="left_only"].drop(columns=["_merge"]).drop_duplicates(statement_key)

out1=output_path(ROOT,RUN_MODE,"15_prepare_compustat_and_ccm","data/processed/stock/compustat_annual_clean.parquet")
out2=output_path(ROOT,RUN_MODE,"15_prepare_compustat_and_ccm","data/processed/stock/ccm_unique_links.parquet")
out3=output_path(ROOT,RUN_MODE,"15_prepare_compustat_and_ccm","data/processed/stock/compustat_ccm_mapped.parquet")
audit=output_path(ROOT,RUN_MODE,"15_prepare_compustat_and_ccm","data/audit/15_ccm_multiple_permno.csv")
write_parquet_atomic(comp,out1); write_parquet_atomic(clean,out2); write_parquet_atomic(mapped_clean,out3); write_csv_atomic(conflicts,audit)
INPUTS_USED=[PATHS["compustat"].name,PATHS["ccm"].name]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out1,out2,out3,audit]]
INPUT_COUNTS={"compustat":raw_n,"ccm":len(ccm)}; OUTPUT_COUNTS={"compustat_clean":len(comp),"ccm_clean":len(clean),"mapped_statements":len(mapped_clean)}
PRIMARY_KEY=["gvkey","datadate"]; DUPLICATE_KEY_COUNT=int(comp.duplicated(["gvkey","datadate"]).sum()); EXCLUDED_ROW_COUNT=len(conflicts); EXCEPTION_FILE=str(audit.relative_to(ROOT))
STAGE_STATUS="passed"; STAGE_NOTES=["CCM uses LPERMNO; at each gvkey+datadate P precedes C, newer LINKDT breaks same-level ties, LC/LU have no priority, unresolved multiple PERMNO are excluded.","B/.B/blank LINKDT and E/.E/blank LINKENDDT are treated as open endpoints.","SH uses SEQ, then CEQ+PS, then AT-LT; only all-missing PS and BE TXDITC use zero."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "15_prepare_compustat_and_ccm", stage="15", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
